# Phase 24 — Contribution arithmetic done in code, not by the model

## The problem

A user asked, in Arabic, what percentage the company pays for an employee's social
security. The app answered **11%**, citing Article 59, and stopped there. The real
employer share is **14.25%**, set by four separate articles.

Phase 23's retrieval work fixed the *context*: sibling expansion now brings all four
funding articles in together. But with all four in front of it, the model still only
reached the right total **3 of 6 times** at one salary and **2 of 6** at another.

| | employee total | employer total |
|---|---|---|
| before any retrieval work | 0/6 | 0/6 |
| after sibling expansion (Phase 23) | 3/6 · 2/6 | 3/6 · 2/6 |

That is a large improvement and still a coin flip on a number someone would act on.

## Why the prompt was never going to close it

To answer "my salary is 1000 JD, what do I pay", the model has to
find six rates spread over four articles → decide which side bears each one → sum
each side → multiply by the wage. Four chained steps, no calculator, at non-zero
temperature. Rules 14–18 were added in Phase 23 telling it to do exactly this, and
3/6 is what instructing a language model to be a calculator buys.

**So the fix is to stop asking it.** `src/calc.py` does the arithmetic; the model is
handed finished figures and writes the answer around them.

The rates are still **read from the retrieved article text**, never hardcoded — if the
corpus is re-extracted or the law is amended, the numbers follow. What is hardcoded is
only *how to read* the text, and every rule below came from a measured failure, not
from assumption.

## Setup

In [1]:
import sys, re
from pathlib import Path
import pandas as pd

PROJECT = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "EXPLAINER.md").exists()), None)
sys.path.insert(0, str(PROJECT))

from src.calc import (article_components, compute_contributions, extract_salary,
                      format_calculation_block, _private_funding_articles)

corpus = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
FAMILY = ["24", "42", "48", "59", "72"]

def art(law, lang, n):
    r = corpus[(corpus.law == law) & (corpus.language == lang) & (corpus.article_number.astype(str) == n)]
    row = r.iloc[0]
    return {"law": law, "language": lang, "article_number": n, "text": str(row["text"]), "chunk_id": row["chunk_id"]}

print(f"corpus: {len(corpus)} articles")

corpus: 531 articles


## Trap 1 — the bearer is named by the verb, not the actor

This is the single thing that makes the split hard to read, for a model or a person.
Look at the two clauses of Article 48:

| clause | text | who really bears it |
|---|---|---|
| 48(أ) | الاشتراكات التي **تؤديها** المنشأة بنسبة (نصف بالمائة) | the employer |
| 48(ب) | الاشتراكات التي **تقتطعها** المنشأة بنسبة (واحد بالمائة) | **the worker** |

The subject is `المنشأة` — "the firm" — in *both*. Only the verb differs:
`تؤدي` (pays) versus `تقتطع` (deducts). English is the same: "paid by the firm"
versus "deducted by the firm".

A first implementation of this module keyed on the actor and tagged all six rates
as employer, producing the right grand total (21.75%) with a completely wrong split.
The cell below shows the raw text so the distinction can be checked, not taken on trust.

In [2]:
for a in ["48", "59"]:
    t = str(corpus[(corpus.law=="ssl") & (corpus.language=="ar") & (corpus.article_number.astype(str)==a)].iloc[0]["text"])
    print(f"--- Article {a}")
    for m in re.finditer(r"(تؤديها|تقتطعها)", t):
        print("   ", t[max(0,m.start()-40): m.start()+90].replace("\n", " "))
    print()

--- Article 48
    مل مما يلي:- أ- الاشتراكات الشهرية التي تؤديها المنشأة بنسبة (نصف بالمائة) من أجور المؤمن عليهم. ب- الاشتراكات الشهرية التي تقتطعه
    لمؤمن عليهم. ب- الاشتراكات الشهرية التي تقتطعها المنشأة بنسبة (واحد بالمائة) من أجور المؤمن عليهم. ج- الفوائد والغرامات التي تترتب

--- Article 59
    اة مما يلي:- ١. الاشتراكات الشهرية التي تؤديها المنشأة بنسبة (١١%) من أجور المؤمن عليهم. ٢. الاشتراكات الشهرية التي تقتطعها المنشأ
    لمؤمن عليهم. ٢. الاشتراكات الشهرية التي تقتطعها المنشأة بنسبة (٦,٥%) من أجور المؤمن عليهم. ٣. الاشتراكات الشهرية التي يؤديها المؤم



## Trap 2 — both verbs sit in the same window

An article states its employer clause and then its employee clause, so any window wide
enough to capture the governing verb also captures the other one. Taking the first verb
found collapses the whole split onto one side.

The clause that owns a rate is the **nearest verb to its left**.

In [3]:
comps = article_components(str(corpus[(corpus.law=="ssl") & (corpus.language=="ar")
                                      & (corpus.article_number.astype(str)=="48")].iloc[0]["text"]))
for c in comps:
    print(f"  {c['rate']:>5}%   bearer={c['bearer']}   scope={c['scope']}")

    0.5%   bearer=employer   scope=private
    1.0%   bearer=employee   scope=private


## Trap 3 — the funding family is not homogeneous

Sibling expansion groups `{24, 42, 48, 59, 72}` because all five share the marker
`تتكون مصادر تمويل`. But **Article 72 is the public-sector and military regime** — an
*alternative* to the other four, not a fifth component of the same total. It carries its
own 11% and 6.5%, numerically identical to the private-sector ones, plus 17%, 21.5% and
26%. Article 59 additionally carries 17.5% (voluntary members) and 50% (a reduction cap).

Counting every percentage in the retrieved family puts **16 figures** in front of the
model when exactly **6** answer the question — and several decoys are indistinguishable
from the right answers by value alone.

In [4]:
rows = []
for a in FAMILY:
    for c in article_components(str(corpus[(corpus.law=="ssl") & (corpus.language=="ar")
                                           & (corpus.article_number.astype(str)==a)].iloc[0]["text"])):
        rows.append({"article": a, **c,
                     "counted": c["scope"] == "private" and c["bearer"] is not None})
df = pd.DataFrame(rows)
print(f"percentages in the retrieved family: {len(df)}   counted: {int(df.counted.sum())}")
display(df)

percentages in the retrieved family: 16   counted: 6


,article,rate,bearer,scope,counted
0,24,2.00,employer,private,True
1,42,0.75,employer,private,True
2,48,0.50,employer,private,True
3,48,1.00,employee,private,True
4,59,11.00,employer,private,True
5,59,6.50,employee,private,True
6,59,17.50,employer,voluntary,False
7,59,50.00,NaN,private,False
8,72,2.00,employer,public,False
9,72,17.00,employer,public,False


`_build_families` in `src/retrieval.py` therefore groups members **by scope** and promotes
only the group the retrieved article belongs to: a private-sector question gets
`{24, 42, 48, 59}`, and a military one gets 72 alone.

In [5]:
from src.retrieval import Retriever
r = object.__new__(Retriever)          # pure function of the corpus -- no index needed
r.chunks = corpus
for key, groups in sorted(Retriever._build_families(r).items()):
    print(key, "->", [sorted(g, key=int) for g in groups])

('ssl', 'ar') -> [['24', '42', '48', '59']]
('ssl', 'en') -> [['24', '42', '48', '59']]


## The completeness guard

Computing from a partial family would reproduce the original bug in a more confident
voice — Article 59 alone yields 11%/6.5%, which is wrong but looks authoritative.

`compute_contributions` returns `None` unless **every** private-scope funding article is
present, leaving the model to answer unaided. The expected set is derived from the
corpus, not written down.

In [6]:
for lang in ("ar", "en"):
    print(lang, "private funding family ->", sorted(_private_funding_articles(lang), key=int))

full = [art("ssl", "ar", a) for a in ["24", "42", "48", "59"]]
print("\ncomplete family      ->", "computed" if compute_contributions(full) else "None")
print("Article 59 alone     ->", "computed" if compute_contributions([art("ssl","ar","59")]) else "None")
print("missing Article 42   ->", "computed" if compute_contributions([art("ssl","ar",a) for a in ["24","48","59"]]) else "None")

ar private funding family -> ['24', '42', '48', '59']
en private funding family -> ['24', '42', '48', '59']

complete family      -> computed
Article 59 alone     -> None
missing Article 42   -> None


## The result, checked against external ground truth

The Social Security Corporation publishes the split as **employee 7.5% · employer 14.25%
· total 21.75%**. Nothing in this module was fitted to those numbers — they are reached
from the article text alone, and they agree in both languages.

In [7]:
for lang in ("ar", "en"):
    res = compute_contributions([art("ssl", lang, a) for a in ["24","42","48","59","72"]])
    print(f"{lang}:  employee {res['employee_rate']}%   employer {res['employer_rate']}%   "
          f"total {res['total_rate']}%   (from articles {sorted({c['article'] for c in res['components']}, key=int)})")
    assert (res["employee_rate"], res["employer_rate"]) == (7.5, 14.25)
print("\nboth languages match the published SSC split, with Article 72 in context and excluded")

ar:  employee 7.5%   employer 14.25%   total 21.75%   (from articles ['24', '42', '48', '59'])


en:  employee 7.5%   employer 14.25%   total 21.75%   (from articles ['24', '42', '48', '59'])

both languages match the published SSC split, with Article 72 in context and excluded


### Rounding: half-up, not half-even

The first paid run returned **121.12 JD** for an 850 JD wage in 3 of 3 trials — faithfully,
because the computed block said 121.12. `850 × 14.25% = 121.125` exactly, and Python's
`round()` rounds half to *even*. Every payslip convention rounds half *up*, to 121.13.

The model was not wrong; the calculator was. Money now goes through `Decimal` with
`ROUND_HALF_UP`, and **every component carries its own precomputed amount** so the model
never multiplies either.

In [8]:
print("python round():", round(850 * 14.25 / 100, 2), "  <- half to even")
res = compute_contributions([art("ssl","en",a) for a in ["24","42","48","59"]], salary=850)
print("calc.py        :", res["employer_amount"], "  <- half up")
print()
print(format_calculation_block(res, "en"))

python round(): 121.12   <- half to even
calc.py        : 121.13   <- half up

[COMPUTED FIGURES (arithmetic already done for you -- do not recalculate)]
Each rate below was read from the article text shown above. Use these figures as given, cite the article beside each one, and do not add, re-derive or adjust any of them.

Deducted from the employee:
  Article (48): 1% = 8.5 JD
  Article (59): 6.5% = 55.25 JD
  Employee total: 7.5% = 63.75 JD

Paid by the employer:
  Article (24): 2% = 17 JD
  Article (42): 0.75% = 6.38 JD
  Article (48): 0.5% = 4.25 JD
  Article (59): 11% = 93.5 JD
  Employer total: 14.25% = 121.13 JD

Combined total: 21.75% = 184.88 JD
(on a monthly wage of 850 JD)


### The wage ceiling

Article 20(أ) caps the contributory wage at 3,000 JD. It is applied **only if Article 20 was
actually retrieved** — not retrieved means not stated, and inventing 3,000 would be exactly
the kind of ungrounded number this whole system exists to avoid.

In [9]:
with20 = [art("ssl","ar",a) for a in ["20","24","42","48","59"]]
hi = compute_contributions(with20, salary=5000)
print(f"5000 JD, Art 20 retrieved     -> base {hi['base']:g}, capped={hi['capped']}, "
      f"employee {hi['employee_amount']}, employer {hi['employer_amount']}")
lo = compute_contributions(with20, salary=1000)
print(f"1000 JD, Art 20 retrieved     -> base {lo['base']:g}, capped={lo['capped']}, "
      f"employee {lo['employee_amount']}, employer {lo['employer_amount']}")
no = compute_contributions([art("ssl","ar",a) for a in ["24","42","48","59"]], salary=5000)
print(f"5000 JD, Art 20 NOT retrieved -> ceiling {no['ceiling']} (never invented)")

5000 JD, Art 20 retrieved     -> base 3000, capped=True, employee 225.0, employer 427.5
1000 JD, Art 20 retrieved     -> base 1000, capped=False, employee 75.0, employer 142.5


5000 JD, Art 20 NOT retrieved -> ceiling None (never invented)


## Offline test suite

`src/test_calc.py` asserts all of the above plus salary parsing (Arabic-Indic digits,
thousands separators) and that no decoy reaches the rendered block. It costs nothing to
run — no API calls — so it can gate every future change to the corpus or the parser.

In [10]:
import subprocess
print(subprocess.run([sys.executable, "-m", "src.test_calc"], cwd=PROJECT,
                     capture_output=True, text=True, encoding="utf-8").stdout[-1400:])

تم، لا تعِد حسابه)]
كل نسبة أدناه مقروءة من نص المواد المعروضة أعلاه. استخدم هذه الأرقام كما هي، واذكر المادة بجانب كل نسبة، ولا تجمع أو تعدّل أياً منها.

يُقتطع من المؤمن عليه:
  المادة (48): 1% = 10 دينار
  المادة (59): 6.5% = 65 دينار
  مجموع ما يُقتطع من المؤمن عليه: 7.5% = 75 دينار

تؤديه المنشأة:
  المادة (24): 2% = 20 دينار
  المادة (42): 0.75% = 7.5 دينار
  المادة (48): 0.5% = 5 دينار
  المادة (59): 11% = 110 دينار
  مجموع ما تؤديه المنشأة: 14.25% = 142.5 دينار

المجموع الكلي: 21.75% = 217.5 دينار
(على أجر شهري مقداره 1000 دينار)
الحد الأعلى للأجر الخاضع للاقتطاع في المادة (20) هو 3000 دينار، وهو لا يؤثر على هذا الحساب.

[COMPUTED FIGURES (arithmetic already done for you -- do not recalculate)]
Each rate below was read from the article text shown above. Use these figures as given, cite the article beside each one, and do not add, re-derive or adjust any of them.

Deducted from the employee:
  Article (48): 1% = 30 JD
  Article (59): 6.5% = 195 JD
  Employee total: 7.5% = 225 JD

## Measured end to end — RETRIEVAL

Production path, all 60 `eval_v3` questions, compared against the Phase 20 saved run.
Reported separately from the generation results below.

| config | recall@1 | recall@5 | MRR | full coverage |
|---|---|---|---|---|
| Phase 20 baseline | 0.867 | 1.000 | 0.922 | 0.950 |
| **now** | **0.900** | **1.000** | **0.939** | **0.950** |

- **0 questions lost coverage**, 0 gained; worst rank 4
- **0 questions now receive Article 72** (was being promoted into private-sector questions)

### A defect this run exposed

`_expand_families` scanned the *entire* candidate list, so a family member at rank 40 —
an article the question was never going to show — still widened the window to
`FAMILY_TOP_K`. It now only fires when the trigger sits within `top_k`:

| | expansion fired | mean slots | retrieval metrics |
|---|---|---|---|
| before | 37 / 60 | 6.85 | — |
| after | **2 / 60** | **5.10** | **identical** |

Identical metrics means the extra context was pure waste — tokens spent on siblings of an
article nobody was going to read.

## Measured end to end — GENERATION

Repeated trials, because temperature is non-zero: one correct answer says "possible",
never "fixed". Answers were persisted to disk, so a scoring fix never costs another
paid run — the lesson from Phases 8 and 20.

| case | expected | before | after |
|---|---|---|---|
| original bug question | employer rate 14.25% | answered 11% | **6/6** |
| salary 1000 JD (ar) | 75 · 142.5 | 3/6 · 3/6 | **6/6 · 6/6** |
| salary 400 JD (ar) | 30 · 57 | 2/6 · 2/6 | **6/6 · 6/6** |
| salary 850 JD (en) | 63.75 · 121.13 | — | **3/3 · 3/3** |
| salary 5000 JD (ar) | 225 · 427.5, capped | — | **3/3 · 3/3** |

- **0 trials reverted to the Article 59-only answer** (the original bug), across all 21+6 runs
- At 5000 JD the uncapped figures (375 / 712.5) **never appeared**, and Article 20's ceiling
  was stated in 3/3

## What this does NOT fix

End-of-service gratuity, overtime rates, leave accrual and the pension formula are still
the model's own arithmetic. The **pattern** generalises — extract in code, compute in code,
let the model narrate — but this extractor is specific to contribution rates. Each further
calculation type is separate work.

Rules 14–18 remain in `SYSTEM_PROMPT` for the cases the calculator declines (an incomplete
family, a calculation it does not cover), and rule 19 tells the model to use the computed
block verbatim where one is present.

**Not measured here:** the effect of rule 19 and the retrieval changes on LLM-judge
faithfulness/relevancy across the full 60-question set. That needs a separate paid judge
run and is not claimed.